# Spaceship Titanic: leakage-audited model research

A reproducible experiment notebook for genuine generalization. All model comparisons use the same stratified ten-fold split (seed 42). Group outcome rates are tested with strict fold-safe, leave-one-out construction; a passenger's own label never enters their feature. Model and threshold selection use out-of-fold predictions only, never the public leaderboard.

The 10-fold model zoo, ablations, and optional Optuna search can take substantial time. Start with the fixed models; enable the bounded search cell when ready.


## 1. Imports and setup


In [ ]:
from pathlib import Path
import importlib.util, subprocess, sys, warnings
packages = {"xgboost":"xgboost", "lightgbm":"lightgbm", "catboost":"catboost", "optuna":"optuna"}
for package, module in packages.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from threadpoolctl import threadpool_limits
threadpool_limits(limits=2)
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import make_column_selector
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
N_SPLITS = 10
np.random.seed(RANDOM_STATE)
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 150)


## 2. Load and combine train/test
The resolver prefers data/raw and also supports the root-level filenames currently present in this repository. The train/test marker only controls splitting; labels are not copied into shared features.


In [ ]:
PROJECT_DIR = Path(r"C:\Users\Administrator\OneDrive\GitHub\spaceshiptitanic")
DATA_DIR = PROJECT_DIR / "data" / "raw"
SUBMISSION_DIR = PROJECT_DIR / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

def resolve_csv(filename, fallback):
    for p in (DATA_DIR / filename, PROJECT_DIR / fallback):
        if p.exists():
            if p.parent == PROJECT_DIR and p.name != filename:
                print(f"Using root-level data file: {p.name}")
            return p
    raise FileNotFoundError(f"Missing {DATA_DIR / filename}")

train = pd.read_csv(resolve_csv("train.csv", "train (1).csv"))
test = pd.read_csv(resolve_csv("test.csv", "test (1).csv"))
sample = pd.read_csv(resolve_csv("sample_submission.csv", "sample_submission.csv"))
target = train["Transported"].astype(int).reset_index(drop=True)
test_ids = test["PassengerId"].copy()
train["_is_train"] = True
test["_is_train"] = False
raw = pd.concat([train, test], ignore_index=True, sort=False)
print("train:", train.shape, "test:", test.shape, "combined:", raw.shape)
display(sample.head())


## 3. Dataset forensics
Inspect types, uniqueness, missingness, duplicates, target balance, and train/test distribution shifts before deriving features. Large train/test shifts are descriptive; they do not authorize target-based changes.


In [ ]:
display(pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "unique_non_null": raw.nunique(dropna=True),
    "missing_count": raw.isna().sum(),
    "missing_pct": (100 * raw.isna().mean()).round(2),
}))
print("Exact duplicate train rows:", train.drop(columns="_is_train", errors="ignore").duplicated().sum())
print("Duplicate PassengerIds train/test:", train.PassengerId.duplicated().sum(), test.PassengerId.duplicated().sum())
display(target.value_counts().rename(index={0:"False",1:"True"}).to_frame("count").assign(share=lambda d:d["count"]/len(target)))


### Numerical distributions, outliers, and shifts
Compare continuous fields by split, inspect robust quantiles, and visualize skewed spend on a log scale. This identifies distribution and outlier behavior before model choice.


In [ ]:
spend = ["RoomService","FoodCourt","ShoppingMall","Spa","VRDeck"]
numeric = ["Age"] + spend
display(raw.groupby("_is_train")[numeric].describe(percentiles=[.01,.25,.5,.75,.99]).T)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, numeric):
    sns.histplot(data=raw, x=col, hue="_is_train", bins=35, element="step", stat="density", common_norm=False, ax=ax)
    if col in spend:
        ax.set_xscale("symlog", linthresh=1)
plt.tight_layout()
plt.show()


### Categorical relationships and missingness signal
Outcome summaries are training-only descriptive statistics. Pairwise categorical tables reveal dependencies. Missingness rates are compared by class to decide whether indicator features deserve inclusion.


In [ ]:
cat_raw = ["HomePlanet","CryoSleep","Cabin","Destination","VIP"]
for col in cat_raw:
    display(train.groupby(col, dropna=False)["Transported"].agg(["mean","count"]).sort_values("count", ascending=False).head(20))
missingness = pd.DataFrame({
    "missing_pct": 100 * train.drop(columns=["Transported","_is_train"], errors="ignore").isna().mean(),
    "target_when_missing": [train.loc[train[c].isna(),"Transported"].mean() for c in train.drop(columns=["Transported","_is_train"], errors="ignore").columns],
    "target_when_present": [train.loc[train[c].notna(),"Transported"].mean() for c in train.drop(columns=["Transported","_is_train"], errors="ignore").columns],
})
display(missingness.sort_values("missing_pct", ascending=False))


## 4. Group and cabin reconstruction
PassengerId groups and nearby cabins expose shared context. These engineered values use passenger attributes only, never Transported. In CV a held-out passenger may use the other observed attributes of their travel group, just as those attributes are available at inference; no validation label crosses folds.


In [ ]:
SPEND = ["RoomService","FoodCourt","ShoppingMall","Spa","VRDeck"]

def engineer(frame):
    f = frame.copy()
    for c in ["HomePlanet","CryoSleep","Cabin","Destination","Age","VIP",*SPEND,"Name"]:
        f[c + "Missing"] = f[c].isna().astype("int8")
    ids = f.PassengerId.str.split("_", expand=True)
    f["GroupId"] = ids[0]
    f["PassengerNumber"] = pd.to_numeric(ids[1], errors="coerce")
    f["GroupSize"] = f.groupby("GroupId").PassengerId.transform("size")
    f["IsSingleton"] = (f.GroupSize == 1).astype("int8")
    f["IsLargeGroup"] = (f.GroupSize >= 5).astype("int8")
    cabin = f.Cabin.str.split("/", expand=True)
    f["CabinMalformed"] = f.Cabin.notna() & (cabin[0].isna() | pd.to_numeric(cabin[1],errors="coerce").isna() | cabin[2].isna())
    f["Deck"], f["CabinNum"], f["Side"] = cabin[0], pd.to_numeric(cabin[1], errors="coerce"), cabin[2]
    f["Surname"] = f.Name.str.rsplit(n=1).str[-1].fillna("Unknown")
    f["FamilySize"] = f.groupby(["GroupId","Surname"]).PassengerId.transform("size")
    for c in ["HomePlanet","Destination","CryoSleep","VIP","Deck","Side"]:
        mode = f.groupby("GroupId")[c].transform(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
        f[c] = f[c].fillna(mode).fillna("Unknown")
    f["GroupAgeMean"] = f.groupby("GroupId").Age.transform("mean")
    f["GroupAgeMedian"] = f.groupby("GroupId").Age.transform("median")
    f["GroupCryoUniform"] = f.groupby("GroupId").CryoSleep.transform(lambda s: int(s.nunique(dropna=False)==1))
    f["GroupPlanetUniform"] = f.groupby("GroupId").HomePlanet.transform(lambda s: int(s.nunique(dropna=False)==1))
    f["GroupDestinationUniform"] = f.groupby("GroupId").Destination.transform(lambda s: int(s.nunique(dropna=False)==1))
    f["GroupVIPCount"] = f.groupby("GroupId").VIP.transform(lambda s: (s.astype(str)=="True").sum())
    f["DeckSide"] = f.Deck.astype(str) + "_" + f.Side.astype(str)
    f["DeckPlanet"] = f.Deck.astype(str) + "_" + f.HomePlanet.astype(str)
    f["CryoPlanet"] = f.CryoSleep.astype(str) + "_" + f.HomePlanet.astype(str)
    f["PlanetDestination"] = f.HomePlanet.astype(str) + "_" + f.Destination.astype(str)
    f["CabinNum"] = f.CabinNum.fillna(f.groupby("GroupId").CabinNum.transform("median"))
    f["CabinNum"] = f.CabinNum.fillna(f.CabinNum.median())
    f["CabinRegion"] = pd.cut(f.CabinNum, [-np.inf,300,700,np.inf], labels=["front","middle","rear"]).astype(object)
    f["CabinBin"] = pd.qcut(f.CabinNum.rank(method="first"), 10, labels=False, duplicates="drop")
    f["DeckCabinPct"] = f.groupby("Deck").CabinNum.rank(pct=True)
    f["Age"] = f.Age.fillna(f.groupby("GroupId").Age.transform("median")).fillna(f.Age.median())
    f["GroupAgeMean"] = f["GroupAgeMean"].fillna(f["Age"].median())
    f["GroupAgeMedian"] = f["GroupAgeMedian"].fillna(f["Age"].median())
    f["AgeGroup"] = pd.cut(f.Age, [-np.inf,12,17,64,np.inf], labels=["child","teen","adult","senior"]).astype(object)
    for c in SPEND:
        f[c] = f[c].fillna(f.groupby("GroupId")[c].transform("median"))
        f[c] = f[c].fillna(f[c].median()).fillna(0)
    spend_matrix = f[SPEND]
    f["TotalSpend"] = spend_matrix.sum(axis=1)
    f["SpendUsedCount"] = (spend_matrix > 0).sum(axis=1)
    f["ZeroSpendCount"] = (spend_matrix == 0).sum(axis=1)
    f["MaxSpend"] = spend_matrix.max(axis=1)
    f["MinNonZeroSpend"] = spend_matrix.replace(0,np.nan).min(axis=1).fillna(0)
    f["MeanSpend"] = spend_matrix.mean(axis=1)
    f["MedianSpend"] = spend_matrix.median(axis=1)
    f["SpendVariance"] = spend_matrix.var(axis=1)
    f["SpendConcentration"] = f["MaxSpend"] / (f["TotalSpend"] + 1)
    f["AnySpend"] = (f.TotalSpend > 0).astype("int8")
    f["CryoSpendContradiction"] = (f.CryoSleep.astype(str)=="True") & (f.TotalSpend>0)
    f["AwakeNoSpend"] = (f.CryoSleep.astype(str)=="False") & (f.TotalSpend==0)
    for c in SPEND:
        f[c+"Log"] = np.log1p(f[c])
        f[c+"Flag"] = (f[c]>0).astype("int8")
        f[c+"Ratio"] = f[c] / (f.TotalSpend+1)
        f[c+"PerGroupMember"] = f[c] / f.GroupSize
    f["TotalSpendPerGroupMember"] = f.TotalSpend / f.GroupSize
    f["LuxurySpend"] = (f.Spa + f.VRDeck > 0).astype("int8")
    f["SpendByCryo"] = f.TotalSpend * f.CryoSleep.astype(str).eq("True")
    f["SpendByVIP"] = f.TotalSpend * f.VIP.astype(str).eq("True")
    interaction_levels = {"Planet":["Earth","Europa","Mars","Unknown"], "Destination":["TRAPPIST-1e","55 Cancri e","PSO J318.5-22","Unknown"], "AgeGroup":["child","teen","adult","senior"], "Deck":["A","B","C","D","E","F","G","T","Unknown"], "Side":["P","S","Unknown"]}
    interaction_columns = {"Planet":"HomePlanet","Destination":"Destination","AgeGroup":"AgeGroup","Deck":"Deck","Side":"Side"}
    for label, levels in interaction_levels.items():
        category = interaction_columns[label]
        for level in levels:
            f[f"SpendBy{label}_{level}"] = f.TotalSpend * f[category].astype(str).eq(level)
    return f.drop(columns=["PassengerId","Name","Cabin","GroupId","_is_train","Transported"], errors="ignore")

features_all = engineer(raw)
is_train = raw["_is_train"].to_numpy()
X = features_all.loc[is_train].reset_index(drop=True)
X_test = features_all.loc[~is_train].reset_index(drop=True)
for col in X.columns:
    if X[col].dtype == "object" or str(X[col].dtype) == "category" or X[col].dtype == "bool":
        X[col] = X[col].astype(str)
        X_test[col] = X_test[col].astype(str)
assert X.isna().sum().sum() == X_test.isna().sum().sum() == 0
print("feature matrices:", X.shape, X_test.shape)


## Group and cabin outcome diagnostics
This section measures correlations and within-group outcome agreement for understanding only. It does not turn group labels into model features. That avoids target leakage from a group member placed in another fold.


In [ ]:
group_ids = train.PassengerId.str.split("_").str[0]
g = train.assign(GroupId=group_ids).groupby("GroupId").Transported
group_table = g.agg(size="size", rate="mean")
multi = group_table[group_table["size"]>1]
print("Multi-person groups:",len(multi),"pure outcome groups:",round(multi.rate.isin([0,1]).mean(),4))
display(train.assign(Deck=train.Cabin.str.split("/").str[0], Side=train.Cabin.str.split("/").str[-1])
        .groupby(["Deck","Side"],dropna=False).Transported.agg(["mean","count"]))

# Outcome correlation among nearby cabin numbers is descriptive only; it never enters features.
nearby=train.assign(Deck=train.Cabin.str.split("/").str[0], CabinNum=pd.to_numeric(train.Cabin.str.split("/").str[1],errors="coerce"))
nearby=nearby.dropna(subset=["Deck","CabinNum"]).sort_values(["Deck","CabinNum"])
nearby["NearbyOutcomeRate"]=nearby.groupby("Deck").Transported.transform(lambda v:v.rolling(11,center=True,min_periods=3).mean())
print("Nearby cabin rolling outcome correlation:",round(nearby[["NearbyOutcomeRate","Transported"]].corr().iloc[0,1],4))


## 5. Feature families and missing-value strategy
The model pipeline compares feature families explicitly. Engineered group imputation is hierarchical (group first, global fallback); all missingness indicators remain available. No row is dropped. Base feature columns below provide controlled ablations.


In [ ]:
BASE = ["HomePlanet","CryoSleep","Destination","Age","VIP",*SPEND]
CABIN = ["Deck","CabinNum","Side","CabinRegion","CabinBin","DeckCabinPct","DeckSide","DeckPlanet","CabinMalformed"]
GROUP = ["PassengerNumber","GroupSize","IsSingleton","IsLargeGroup","Surname","FamilySize","GroupAgeMean","GroupAgeMedian","GroupCryoUniform","GroupPlanetUniform","GroupDestinationUniform","GroupVIPCount"]
SPEND_FEATURES = [c for c in X.columns if c in SPEND or c.startswith(tuple(SPEND)) or c in ["TotalSpend","SpendUsedCount","ZeroSpendCount","MaxSpend","MinNonZeroSpend","MeanSpend","MedianSpend","SpendVariance","SpendConcentration","AnySpend","CryoSpendContradiction","AwakeNoSpend","LuxurySpend","TotalSpendPerGroupMember","SpendByCryo","SpendByVIP"]]
MISSING = [c for c in X.columns if c.endswith("Missing")]
INTERACTIONS = ["CryoPlanet","PlanetDestination"] + [c for c in X.columns if c.startswith(("SpendByPlanet_","SpendByDestination_","SpendByAgeGroup_","SpendByDeck_","SpendBySide_"))]
families = {
    "Baseline": [c for c in BASE if c in X],
    "Baseline + Cabin": list(dict.fromkeys([*BASE,*CABIN])),
    "Baseline + Spending": list(dict.fromkeys([*BASE,*SPEND_FEATURES])),
    "Baseline + Group": list(dict.fromkeys([*BASE,*GROUP])),
    "Baseline + Missingness": list(dict.fromkeys([*BASE,*MISSING])),
    "Baseline + Interactions": list(dict.fromkeys([*BASE,*INTERACTIONS])),
    "Full": list(X.columns),
}
families = {k:[c for c in v if c in X.columns] for k,v in families.items()}
print({k:len(v) for k,v in families.items()})


## 6. Shared ten-fold CV and preprocessing
The exact same folds are reused throughout. Low-cardinality categoricals are one-hot encoded; numerics are median-imputed and scaled only for Logistic Regression. This cell also defines score reporting with mean, standard deviation, minimum, and maximum fold accuracy.


In [ ]:
folds = list(StratifiedKFold(n_splits=10,shuffle=True,random_state=42).split(X,target))
results = {}
def make_prep(cols, scale=False, dense=False):
    cats = [c for c in cols if X[c].dtype=="object" or str(X[c].dtype)=="category" or X[c].dtype=="bool"]
    nums = [c for c in cols if c not in cats]
    try: ohe=OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=15, max_categories=64, sparse_output=dense is False)
    except TypeError: ohe=OneHotEncoder(handle_unknown="ignore", sparse=dense is False)
    numeric_steps=[("impute",SimpleImputer(strategy="median"))]
    if scale: numeric_steps.append(("scale",StandardScaler()))
    return ColumnTransformer([("cat",ohe,cats),("num",Pipeline(numeric_steps),nums)],remainder="drop",sparse_threshold=1.0)
def summarize(name, oof, split_list=folds):
    scores=[accuracy_score(target.iloc[va],oof[va]>=.5) for _,va in split_list]
    ans={"name":name,"oof":oof,"mean":float(np.mean(scores)),"std":float(np.std(scores)),"min":float(np.min(scores)),"max":float(np.max(scores)),"folds":scores}
    print(f"{name}: mean={ans['mean']:.5f} std={ans['std']:.5f} min={ans['min']:.5f} max={ans['max']:.5f}")
    return ans
def evaluate(name, factory, cols=None, data=X):
    cols=cols or list(data.columns)
    oof=np.zeros(len(target))
    for i,(tr,va) in enumerate(folds,1):
        model=factory(cols)
        y_fit=np.ascontiguousarray(target.iloc[tr].to_numpy(),dtype=np.int32) if name=="LightGBM" else target.iloc[tr]
        model.fit(data.iloc[tr][cols],y_fit)
        oof[va]=model.predict_proba(data.iloc[va][cols])[:,1]
        print(f"{name} fold {i}/10",flush=True)
    results[name]=summarize(name,oof)
    return results[name]


## 7. Model zoo baseline
Evaluate Logistic Regression, Random Forest, Extra Trees, HistGradientBoosting, XGBoost, LightGBM, and CatBoost on the full feature set. CatBoost consumes categorical columns natively. HistGradientBoosting uses the encoded matrix because it has no direct string-category support in this shared pipeline.


In [ ]:
def sklearn_model(kind, cols):
    prep=make_prep(cols,scale=(kind=="Logistic Regression"),dense=(kind in ["HistGradientBoosting","LightGBM"]))
    if kind=="Logistic Regression":
        estimator=LogisticRegression(C=0.5,solver="liblinear",max_iter=120,tol=1e-2,random_state=42)
    elif kind=="Random Forest":
        estimator=RandomForestClassifier(n_estimators=500,min_samples_leaf=2,max_features="sqrt",n_jobs=4,random_state=42)
    elif kind=="Extra Trees":
        estimator=ExtraTreesClassifier(n_estimators=500,min_samples_leaf=2,max_features="sqrt",n_jobs=4,random_state=42)
    elif kind=="HistGradientBoosting":
        estimator=HistGradientBoostingClassifier(max_iter=250,learning_rate=.06,max_leaf_nodes=15,l2_regularization=2,random_state=42)
    elif kind=="XGBoost":
        estimator=XGBClassifier(n_estimators=450,max_depth=4,learning_rate=.035,min_child_weight=3,subsample=.85,colsample_bytree=.85,reg_lambda=3,eval_metric="logloss",n_jobs=4,random_state=42,tree_method="hist")
    elif kind=="LightGBM":
        estimator=LGBMClassifier(n_estimators=450,num_leaves=15,learning_rate=.03,min_child_samples=25,subsample=.85,colsample_bytree=.85,reg_lambda=2,verbosity=-1,n_jobs=4,random_state=42)
    if kind=="LightGBM":
        return Pipeline([("prep",prep),("contiguous",FunctionTransformer(lambda a:np.ascontiguousarray(a,dtype=np.float32),validate=False)),("model",estimator)])
    return Pipeline([("prep",prep),("model",estimator)])
def cat_factory(cols):
    return CatBoostClassifier(iterations=600,depth=6,learning_rate=.04,l2_leaf_reg=5,loss_function="Logloss",verbose=False,allow_writing_files=False,thread_count=4,random_seed=42,cat_features=[c for c in cols if X[c].dtype=="object" or str(X[c].dtype)=="category"])
full_cols=families["Full"]
model_failures={}
for model_name in ["Logistic Regression","Random Forest","Extra Trees","HistGradientBoosting","XGBoost","LightGBM"]:
    try:
        evaluate(model_name,lambda cols,n=model_name:sklearn_model(n,cols),full_cols)
    except Exception as exc:
        model_failures[model_name]=repr(exc)
        print(f"{model_name} unavailable in this runtime: {exc}")
# CatBoost gets raw categorical values, including high-cardinality surname.
cat_cols=[c for c in full_cols if X[c].dtype=="object" or str(X[c].dtype)=="category"]
X_cat=X.copy()
for c in cat_cols: X_cat[c]=X_cat[c].astype(str)
oof=np.zeros(len(target))
for i,(tr,va) in enumerate(folds,1):
    model=cat_factory(full_cols); model.fit(X_cat.iloc[tr][full_cols],target.iloc[tr])
    oof[va]=model.predict_proba(X_cat.iloc[va][full_cols])[:,1]
    print(f"CatBoost fold {i}/10")
results["CatBoost"]=summarize("CatBoost",oof)


## Fold-safe group outcome signal
Group members sometimes share outcomes. This experiment adds smoothed group Transported rate and labeled-member count. In each fold, a training row's rate excludes its own label; validation rows use only labels from that fold's training rows. The final test features use all labeled training passengers. This is an explicitly supervised feature, so fold isolation is essential. Compare its ten-fold result with ordinary CatBoost before selecting it.


In [ ]:
group_key_train = train.PassengerId.str.split("_").str[0].reset_index(drop=True)
group_key_test = test.PassengerId.str.split("_").str[0].reset_index(drop=True)
GROUP_ALPHA = 2.0
group_cat_cols = [c for c in full_cols if X[c].dtype == "object" or str(X[c].dtype) == "category"]
group_model_cols = full_cols + ["GroupOutcomeRate", "GroupLabeledCount"]
group_oof = np.zeros(len(target))

for fold, (tr, va) in enumerate(folds, 1):
    gtr = group_key_train.iloc[tr].reset_index(drop=True)
    ytr = target.iloc[tr].reset_index(drop=True)
    prior = float(ytr.mean())
    sums = ytr.groupby(gtr).transform("sum").to_numpy()
    counts = ytr.groupby(gtr).transform("count").to_numpy()
    train_rate = (sums - ytr.to_numpy() + GROUP_ALPHA * prior) / (counts - 1 + GROUP_ALPHA)
    train_count = counts - 1
    train_fold = X.iloc[tr][full_cols].copy()
    train_fold["GroupOutcomeRate"] = train_rate
    train_fold["GroupLabeledCount"] = train_count

    fold_stats = pd.DataFrame({"group": gtr, "label": ytr}).groupby("group").label.agg(["sum", "count"])
    gva = group_key_train.iloc[va]
    val_count = gva.map(fold_stats["count"]).fillna(0).to_numpy()
    val_sum = gva.map(fold_stats["sum"]).fillna(0).to_numpy()
    val_rate = (val_sum + GROUP_ALPHA * prior) / (val_count + GROUP_ALPHA)
    valid_fold = X.iloc[va][full_cols].copy()
    valid_fold["GroupOutcomeRate"] = val_rate
    valid_fold["GroupLabeledCount"] = val_count

    model = CatBoostClassifier(iterations=600, depth=6, learning_rate=.04, l2_leaf_reg=5,
        loss_function="Logloss", verbose=False, allow_writing_files=False, thread_count=4,
        random_seed=42, cat_features=group_cat_cols)
    model.fit(train_fold[group_model_cols], ytr)
    group_oof[va] = model.predict_proba(valid_fold[group_model_cols])[:, 1]
    print(f"Fold-safe group CatBoost fold {fold}/10", flush=True)

results["CatBoost + fold-safe group target"] = summarize("CatBoost + fold-safe group target", group_oof)

def fit_group_target_catboost_for_test():
    prior = float(target.mean())
    sums = target.groupby(group_key_train).transform("sum").to_numpy()
    counts = target.groupby(group_key_train).transform("count").to_numpy()
    train_rate = (sums - target.to_numpy() + GROUP_ALPHA * prior) / (counts - 1 + GROUP_ALPHA)
    train_count = counts - 1
    full_train = X[full_cols].copy()
    full_train["GroupOutcomeRate"] = train_rate
    full_train["GroupLabeledCount"] = train_count
    stats = pd.DataFrame({"group": group_key_train, "label": target}).groupby("group").label.agg(["sum", "count"])
    test_count = group_key_test.map(stats["count"]).fillna(0).to_numpy()
    test_sum = group_key_test.map(stats["sum"]).fillna(0).to_numpy()
    test_rate = (test_sum + GROUP_ALPHA * prior) / (test_count + GROUP_ALPHA)
    full_test = X_test[full_cols].copy()
    full_test["GroupOutcomeRate"] = test_rate
    full_test["GroupLabeledCount"] = test_count
    model = CatBoostClassifier(iterations=600, depth=6, learning_rate=.04, l2_leaf_reg=5,
        loss_function="Logloss", verbose=False, allow_writing_files=False, thread_count=4,
        random_seed=42, cat_features=group_cat_cols)
    model.fit(full_train[group_model_cols], target)
    return model, full_test[group_model_cols]


## 8. Feature-family ablation
Extra Trees provides a stable reference for comparing cabin, spending, group, missingness, interaction, and full feature families on identical folds. These are measured experiments, not claims that every engineered field helps.


In [ ]:
for family_name, cols in families.items():
    if family_name=="Full": continue
    evaluate("Ablation: "+family_name,lambda c:sklearn_model("Extra Trees",c),cols)
ablation_table=pd.DataFrame([{"Experiment":k,**{x:v[x] for x in ["mean","std","min","max"]}} for k,v in results.items() if k.startswith("Ablation:")])
display(ablation_table.sort_values("mean",ascending=False))


## 9. Bounded hyperparameter search
Optuna tunes high-impact parameters for XGBoost and LightGBM against the same fixed folds. This is a compact controlled search rather than an unbounded sweep. CatBoost settings can be adjusted and rerun in the same pattern after reviewing its baseline.


In [ ]:
OPTUNA_TRIALS=20
def optuna_search(family):
    def objective(trial):
        if family=="XGBoost":
            p={"max_depth":trial.suggest_int("max_depth",3,5),"learning_rate":trial.suggest_float("learning_rate",.02,.08,log=True),"min_child_weight":trial.suggest_int("min_child_weight",2,8),"subsample":trial.suggest_float("subsample",.75,1),"colsample_bytree":trial.suggest_float("colsample_bytree",.7,1),"reg_lambda":trial.suggest_float("reg_lambda",1,12,log=True),"reg_alpha":trial.suggest_float("reg_alpha",1e-3,1,log=True)}
            build=lambda cols:Pipeline([("prep",make_prep(cols)),("model",XGBClassifier(n_estimators=500,eval_metric="logloss",n_jobs=4,random_state=42,tree_method="hist",**p))])
        else:
            p={"num_leaves":trial.suggest_int("num_leaves",8,24),"learning_rate":trial.suggest_float("learning_rate",.02,.08,log=True),"min_child_samples":trial.suggest_int("min_child_samples",15,50),"subsample":trial.suggest_float("subsample",.75,1),"colsample_bytree":trial.suggest_float("colsample_bytree",.7,1),"reg_lambda":trial.suggest_float("reg_lambda",1,12,log=True)}
            build=lambda cols:Pipeline([("prep",make_prep(cols,dense=True)),("contiguous",FunctionTransformer(lambda a:np.ascontiguousarray(a,dtype=np.float32),validate=False)),("model",LGBMClassifier(n_estimators=500,verbosity=-1,n_jobs=4,random_state=42,**p))])
        pred=np.zeros(len(target))
        for tr,va in folds:
            m=build(full_cols); m.fit(X.iloc[tr][full_cols],np.ascontiguousarray(target.iloc[tr].to_numpy(),dtype=np.int32) if family=="LightGBM" else target.iloc[tr]); pred[va]=m.predict_proba(X.iloc[va][full_cols])[:,1]
        return np.mean([accuracy_score(target.iloc[va],pred[va]>=.5) for _,va in folds])
    study=optuna.create_study(direction="maximize",sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective,n_trials=OPTUNA_TRIALS,show_progress_bar=True)
    print(f"{family} best mean CV accuracy={study.best_value:.5f}; params={study.best_params}")
    return study
xgb_study=optuna_search("XGBoost")
lgb_study=optuna_search("LightGBM") if "LightGBM" in results else None


def tuned_xgb(cols):
    params=xgb_study.best_params
    model=XGBClassifier(n_estimators=500,eval_metric="logloss",n_jobs=4,random_state=42,tree_method="hist",**params)
    return Pipeline([("prep",make_prep(cols)),("model",model)])
def tuned_lgb(cols):
    params=lgb_study.best_params
    model=LGBMClassifier(n_estimators=500,verbosity=-1,n_jobs=4,random_state=42,**params)
    return Pipeline([("prep",make_prep(cols,dense=True)),("contiguous",FunctionTransformer(lambda a:np.ascontiguousarray(a,dtype=np.float32),validate=False)),("model",model)])
evaluate("XGBoost tuned",tuned_xgb,full_cols)
if lgb_study is not None:
    evaluate("LightGBM tuned",tuned_lgb,full_cols)


## 10. OOF ensemble and threshold selection
Compare single models, pairwise and three-model probability averages, weighted averages, and a logistic stack trained on out-of-fold probabilities. The stack's meta learner is evaluated with a second stratified CV over OOF features; the selected threshold is optimized on OOF probabilities only and is reported separately from fixed-threshold fold results.


In [ ]:
base_names=[n for n in ["XGBoost","LightGBM","CatBoost","Extra Trees"] if n in results]
blend_results={}
for k in [2,3]:
    names=base_names[:k]
    p=np.mean([results[n]["oof"] for n in names],axis=0)
    blend_results["Mean "+ "+".join(names)]=summarize("Mean "+ "+".join(names),p)
# Weight search uses OOF predictions only.
best_weight=(None,-1)
if len(base_names)>=2:
    a,b=base_names[:2]
    for w in np.linspace(0,1,101):
        p=w*results[a]["oof"]+(1-w)*results[b]["oof"]
        score=accuracy_score(target,p>=.5)
        if score>best_weight[1]: best_weight=((a,b,float(w)),score)
    names_a,names_b,w=best_weight[0]
    p=w*results[names_a]["oof"]+(1-w)*results[names_b]["oof"]
    blend_results[f"Weighted {names_a}+{names_b}"]=summarize(f"Weighted {names_a}+{names_b}",p)
# Stacking features are OOF probabilities; assess the meta-classifier via CV on those features.
meta_X=np.column_stack([results[n]["oof"] for n in base_names])
meta_oof=np.zeros(len(target))
for tr,va in folds:
    meta=LogisticRegression(C=.5,max_iter=1000,random_state=42).fit(meta_X[tr],target.iloc[tr])
    meta_oof[va]=meta.predict_proba(meta_X[va])[:,1]
blend_results["OOF logistic stack"]=summarize("OOF logistic stack",meta_oof)
results.update(blend_results)
valid_names=[n for n in results if n in ["Logistic Regression","Random Forest","Extra Trees","HistGradientBoosting","XGBoost","LightGBM","CatBoost","XGBoost tuned","LightGBM tuned"] or n.startswith("Mean ") or n.startswith("Weighted ")]
best_prob_name=max(valid_names,key=lambda n:accuracy_score(target,results[n]["oof"]>=.5))
best_oof=results[best_prob_name]["oof"]
thresholds=np.arange(.35,.651,.005)
threshold_scores=[accuracy_score(target,best_oof>=t) for t in thresholds]
best_threshold=float(thresholds[int(np.argmax(threshold_scores))])
print("OOF-selected threshold:",best_threshold,"OOF accuracy:",max(threshold_scores))


## 11. Threshold and fold robustness
The selected cutoff is chosen from OOF probabilities. The following reports accuracy at that cutoff on each held-out fold, plus the standard 0.5 results, making threshold improvements and fold stability visible.


In [ ]:
chosen=results[best_prob_name]
# Choose a threshold using only each outer training fold's OOF rows, then score on its held-out fold.
threshold_fold=[]
threshold_grid=np.arange(.35,.651,.005)
for tr,va in folds:
    train_scores=[accuracy_score(target.iloc[tr],chosen["oof"][tr]>=t) for t in threshold_grid]
    fold_threshold=threshold_grid[int(np.argmax(train_scores))]
    threshold_fold.append(accuracy_score(target.iloc[va],chosen["oof"][va]>=fold_threshold))
print(f"Threshold CV (per-fold threshold chosen on remaining OOF rows), approach={best_prob_name}")
print(f"Mean={np.mean(threshold_fold):.5f} std={np.std(threshold_fold):.5f} min={np.min(threshold_fold):.5f} max={np.max(threshold_fold):.5f}")


## 12. Feature importance
Fit the strongest individual tree model on all labeled rows and plot native or impurity importance. This is descriptive; ablation CV, not importance rank alone, decides whether feature families help.


In [ ]:
tree_candidates=[n for n in ["Extra Trees","Random Forest","XGBoost","LightGBM","CatBoost","CatBoost + fold-safe group target"] if n in results]
tree_name=max(tree_candidates,key=lambda n:results[n]["mean"])
if tree_name=="CatBoost + fold-safe group target":
    imp_model,_=fit_group_target_catboost_for_test()
    importance=pd.Series(imp_model.get_feature_importance(),index=group_model_cols)
elif tree_name=="CatBoost":
    final_for_importance=cat_factory(full_cols); final_for_importance.fit(X_cat[full_cols],target)
    importance=pd.Series(final_for_importance.get_feature_importance(),index=full_cols)
else:
    final_for_importance=sklearn_model(tree_name,full_cols); final_for_importance.fit(X[full_cols],target)
    fn=final_for_importance.named_steps["prep"].get_feature_names_out()
    importance=pd.Series(final_for_importance.named_steps["model"].feature_importances_,index=fn)
importance.sort_values().tail(25).plot.barh(figsize=(9,8),color="#4E79A7")
plt.title("Feature importance: "+tree_name); plt.tight_layout(); plt.show()
display(importance.sort_values(ascending=False).head(25).to_frame("importance"))


## 13. Leakage audit and final selection
Cabin and raw engineered attributes use only observable columns. The group-target experiment is the only target-derived feature: training rows use leave-one-out group rates; validation rows use labels from that fold's training partition only; final test rates use labeled training passengers only. No passenger's own label contributes to that passenger's group statistic. All CV predictions are out of fold. Train/test concatenation is used only for unlabeled context parsing and counts. Exact PassengerId duplicates are checked above. The OOF stack's reported meta-CV is a useful comparison but may be mildly optimistic because base OOF rows share overlapping training folds; prefer a single model or simple blend unless nested stacking is implemented and validated.


In [ ]:
eligible=[n for n in results if n in ["Logistic Regression","Random Forest","Extra Trees","HistGradientBoosting","XGBoost","LightGBM","CatBoost","XGBoost tuned","LightGBM tuned","CatBoost + fold-safe group target"] or n.startswith("Mean ") or n.startswith("Weighted ")]
candidates={n:results[n] for n in eligible}
final_name=max(candidates,key=lambda n:candidates[n]["mean"])
final_result=candidates[final_name]
final_oof=final_result["oof"]
thresholds=np.arange(.35,.651,.005)
threshold_scores=[accuracy_score(target,final_oof>=t) for t in thresholds]
best_threshold=float(thresholds[int(np.argmax(threshold_scores))])
print(f"Selected by fixed-threshold OOF CV: {final_name}")
print(f"Mean={final_result['mean']:.5f}, std={final_result['std']:.5f}, min={final_result['min']:.5f}, max={final_result['max']:.5f}")
experiment_table=pd.DataFrame([{"Experiment":n,"CV mean":r["mean"],"CV std":r["std"],"Min fold":r["min"],"Max fold":r["max"]} for n,r in results.items()]).sort_values("CV mean",ascending=False)
display(experiment_table)


## 14. Refit and create Kaggle submission
Refit the selected single model on all labeled examples, then predict every test row. When a blend wins, refit its member models and average probabilities. Threshold is selected from OOF predictions only. Validate schema, count, uniqueness, and boolean predictions before writing.


In [ ]:
def full_predict(model_name):
    if model_name=="XGBoost tuned":
        m=tuned_xgb(full_cols); m.fit(X[full_cols],target); return m.predict_proba(X_test[full_cols])[:,1]
    if model_name=="LightGBM tuned":
        m=tuned_lgb(full_cols); m.fit(X[full_cols],target); return m.predict_proba(X_test[full_cols])[:,1]
    if model_name=="CatBoost + fold-safe group target":
        m, final_test=fit_group_target_catboost_for_test(); return m.predict_proba(final_test)[:,1]
    if model_name=="CatBoost":
        m=cat_factory(full_cols); m.fit(X_cat[full_cols],target)
        return m.predict_proba(X_test.assign(**{c:X_test[c].astype(str) for c in cat_cols})[full_cols])[:,1]
    if model_name.startswith("Mean "):
        names=model_name.removeprefix("Mean ").split("+")
        return np.mean([full_predict(n) for n in names],axis=0)
    if model_name.startswith("Weighted "):
        a,b,w=best_weight[0]
        return w*full_predict(a)+(1-w)*full_predict(b)
    if model_name=="OOF logistic stack":
        raise RuntimeError("Select a single model or blend for final refit; the OOF stack requires nested refitting.")
    m=sklearn_model(model_name,full_cols); m.fit(X[full_cols],target)
    return m.predict_proba(X_test[full_cols])[:,1]
if final_name=="OOF logistic stack":
    # Use the best-scoring simple blend/single instead of a non-nested stack for finalization.
    safe=[n for n in candidates if n!="OOF logistic stack"]
    final_name=max(safe,key=lambda n:candidates[n]["mean"])
    print("Stack not selected for deployment because its OOF meta-score is not nested; using",final_name)
test_prob=full_predict(final_name)
submission=pd.DataFrame({"PassengerId":test_ids,"Transported":test_prob>=best_threshold})
assert len(submission)==len(test)
assert submission.PassengerId.notna().all() and submission.PassengerId.is_unique
assert submission.Transported.notna().all() and submission.Transported.isin([True,False]).all()
assert submission.columns.tolist()==["PassengerId","Transported"]
assert submission.PassengerId.equals(sample.PassengerId)
path=SUBMISSION_DIR/"submission_kenga2.csv"
submission.to_csv(path,index=False)
print("Saved:",path,"rows:",len(submission),"threshold:",best_threshold)
display(submission.head())


## Experiment summary


In [ ]:
display(experiment_table.style.format({"CV mean":"{:.5f}","CV std":"{:.5f}","Min fold":"{:.5f}","Max fold":"{:.5f}"}))
print("Final submitted approach:",final_name)
print("Decision threshold:",best_threshold)
print("Submission path:",path)
